# INST-Align visualization example

This notebook runs a small pairwise alignment and reproduces the basic visualization pattern used in the paper: a before/after spatial overlay and a stacked 3D view. Run it from the repository root after installing `requirements.txt` and unzipping the public `Data.zip` archive described in `data/README.md`.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "insta").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "insta").exists():
    raise RuntimeError("Start Jupyter from the repository root or examples/ directory.")

sys.path.insert(0, str(PROJECT_ROOT))
DATA_DIR = PROJECT_ROOT / "Data"
OUTPUT_DIR = PROJECT_ROOT / "examples" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Data directory not found: {DATA_DIR}. "
        "Download and unzip Data.zip as described in data/README.md."
    )

print(f"Project root: {PROJECT_ROOT}")
print(f"Data dir: {DATA_DIR}")
print(f"Output dir: {OUTPUT_DIR}")

In [ ]:
import torch

from insta.config import PipelineConfig, SLICE_ORDER
from insta.metrics import compute_istbench_metrics
from insta.pipeline import align_pair, load_slices, preprocess_slices
from insta.visualization import (
    DLPFC_LAYER_COLORS,
    DLPFC_LAYER_ORDER,
    plot_3d_stack,
    plot_before_after,
)

## Quick 2D alignment example

`STARMap` is used here because it is small and fast enough for a public example. Increase the epoch counts for final-quality runs.

In [ ]:
DATASET = "STARMap"
slice_names = SLICE_ORDER[DATASET][:2]

config = PipelineConfig(
    dataset=DATASET,
    data_dir=str(DATA_DIR),
    output_dir=str(OUTPUT_DIR / "results"),
)
config.device = "cuda" if torch.cuda.is_available() else "cpu"

# Public example settings. Use the defaults in config.py for full paper-scale runs.
config.train.epochs = 10
config.joint.inr_pretrain_epochs = 10
config.train.print_every = 10

print(f"Dataset: {DATASET}")
print(f"Slices: {slice_names}")
print(f"Device: {config.device}")

In [ ]:
slices = load_slices(DATASET, str(DATA_DIR), list(slice_names))
slices = preprocess_slices(slices, n_top_genes=config.n_top_genes, pca_key=config.pca_key)

In [ ]:
aligned_coords, train_result = align_pair(slices[0], slices[1], config, config.device)

aligned_slices = [slices[0].copy(), slices[1].copy()]
aligned_slices[0].obsm["spatial_aligned"] = aligned_slices[0].obsm[config.spatial_key].copy()
aligned_slices[1].obsm["spatial_aligned"] = aligned_coords

metrics_df = compute_istbench_metrics(aligned_slices, list(slice_names), config.label_key)
display(metrics_df)

In [ ]:
plot_before_after(
    slices[0],
    slices[1],
    aligned_src_coords=aligned_coords,
    ref_label=slice_names[0],
    src_label=slice_names[1],
    save_path=OUTPUT_DIR / f"{DATASET.lower()}_before_after.png",
)

## 3D stack view

The same aligned AnnData objects can be rendered as a stacked reconstruction. For DLPFC paper-style coloring, pass `color_map=DLPFC_LAYER_COLORS` and `label_order=DLPFC_LAYER_ORDER`.

In [ ]:
plot_3d_stack(
    aligned_slices,
    label_key=config.label_key,
    spatial_key="spatial_aligned",
    save_path=OUTPUT_DIR / f"{DATASET.lower()}_3d_stack.png",
)

## Optional: DLPFC paper-style colors

After aligning DLPFC slices, call `plot_3d_stack` with the layer color map below.

In [ ]:
# Example for a DLPFC aligned slice list:
# plot_3d_stack(
#     dlpfc_aligned_slices,
#     label_key="original_domain",
#     spatial_key="spatial_aligned",
#     color_map=DLPFC_LAYER_COLORS,
#     label_order=DLPFC_LAYER_ORDER,
#     save_path=OUTPUT_DIR / "dlpfc_3d_stack.png",
# )